In [106]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer


data = pd.read_csv("country_data.csv")
print(data.dtypes)
print(data.isnull().sum())

country        object
child_mort    float64
exports       float64
health        float64
imports       float64
income          int64
inflation     float64
life_expec    float64
total_fer     float64
gdpp            int64
dtype: object
country       0
child_mort    0
exports       0
health        0
imports       0
income        0
inflation     0
life_expec    0
total_fer     0
gdpp          0
dtype: int64


In [107]:
step_num = Pipeline(steps=[
    ('scaler', StandardScaler())
])

step_cat = Pipeline(steps=[
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('angka', step_num, ['child_mort', 'exports', 'health', 'imports', 'income',
       'inflation', 'life_expec', 'total_fer', 'gdpp']),
    ('teks', step_cat, ['country'])
])

master_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=2))
])

df_scaled = master_pipeline.fit_transform(data)
pca_model = master_pipeline.named_steps['pca']
print("\nRasio Informasi yang Dipertahankan:", pca_model.explained_variance_ratio_)


Rasio Informasi yang Dipertahankan: [0.41441261 0.15532644]


In [108]:
best_k = 1
best_score = 0
rentang_k = range(2, 6)

for k in rentang_k:
    model = KMeans(n_clusters=k, random_state=42, n_init='auto')
    label = model.fit_predict(df_scaled)
    score = silhouette_score(df_scaled, label)

    if score > best_score:
        best_score = score
        best_k = k

master_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', KMeans(n_clusters=best_k, random_state=42, n_init='auto'))
])

cluster_labels = master_pipeline.fit_predict(data)
data['Segmen'] = cluster_labels

print(f"Jumlah Cluster Terbaik (K): {best_k} dengan Silhouette Score: {best_score:.4f}")
print(data)

Jumlah Cluster Terbaik (K): 2 dengan Silhouette Score: 0.4138
                 country  child_mort  exports  health  imports  income  \
0            Afghanistan        90.2     10.0    7.58     44.9    1610   
1                Albania        16.6     28.0    6.55     48.6    9930   
2                Algeria        27.3     38.4    4.17     31.4   12900   
3                 Angola       119.0     62.3    2.85     42.9    5900   
4    Antigua and Barbuda        10.3     45.5    6.03     58.9   19100   
..                   ...         ...      ...     ...      ...     ...   
162              Vanuatu        29.2     46.6    5.25     52.7    2950   
163            Venezuela        17.1     28.5    4.91     17.6   16500   
164              Vietnam        23.3     72.0    6.84     80.2    4490   
165                Yemen        56.3     30.0    5.18     34.4    4480   
166               Zambia        83.1     37.0    5.89     30.9    3280   

     inflation  life_expec  total_fer   gdpp  Seg

In [109]:
segmen0 = data.loc[data["Segmen"] == 0]
print(segmen0.describe())

       child_mort    exports    health     imports        income   inflation  \
count   93.000000  93.000000  93.00000   93.000000     93.000000   93.000000   
mean    61.809677  31.448376   5.92957   42.648020   6114.655914   10.850495   
std     40.604511  16.991383   2.36925   18.748384   5769.559516   12.719596   
min     10.000000   0.109000   1.97000    0.065900    609.000000    0.480000   
25%     27.300000  19.700000   4.51000   28.600000   1850.000000    4.150000   
50%     55.200000  27.800000   5.25000   40.300000   4280.000000    7.470000   
75%     90.200000  41.200000   6.85000   53.100000   8460.000000   15.100000   
max    208.000000  85.800000  14.20000  101.000000  33700.000000  104.000000   

       life_expec  total_fer          gdpp  Segmen  
count   93.000000  93.000000     93.000000    93.0  
mean    64.913978   3.867419   2942.301075     0.0  
std      7.858651   1.442296   3057.926382     0.0  
min     32.100000   1.570000    231.000000     0.0  
25%     59.500

In [110]:
segmen1 = data.loc[data["Segmen"] == 1]
print(segmen1.describe())

       child_mort     exports     health     imports         income  \
count   74.000000   74.000000  74.000000   74.000000      74.000000   
mean     8.686486   53.250000   7.929324   52.221622   31006.756757   
std      5.243243   32.810587   2.797414   28.949201   21292.568458   
min      2.600000   10.700000   1.810000   11.800000    3910.000000   
25%      4.225000   32.975000   6.240000   31.700000   15975.000000   
50%      6.950000   48.250000   7.885000   48.250000   27750.000000   
75%     11.650000   66.375000   9.607500   62.900000   41000.000000   
max     23.300000  200.000000  17.900000  174.000000  125000.000000   

       inflation  life_expec  total_fer           gdpp  Segmen  
count  74.000000   74.000000  74.000000      74.000000    74.0  
mean    3.925270   77.645946   1.792432   25559.189189     1.0  
std     4.808578    3.240843   0.398923   21526.128549     0.0  
min    -4.210000   69.700000   1.150000    1310.000000     1.0  
25%     0.827750   75.625000   1.47

In [111]:
print(segmen0["country"].unique())

['Afghanistan' 'Algeria' 'Angola' 'Argentina' 'Armenia' 'Azerbaijan'
 'Bangladesh' 'Belize' 'Benin' 'Bhutan' 'Bolivia' 'Botswana'
 'Burkina Faso' 'Burundi' 'Cambodia' 'Cameroon' 'Cape Verde'
 'Central African Republic' 'Chad' 'China' 'Comoros' 'Congo, Dem. Rep.'
 'Congo, Rep.' "Cote d'Ivoire" 'Dominican Republic' 'Ecuador' 'Egypt'
 'El Salvador' 'Equatorial Guinea' 'Eritrea' 'Fiji' 'Gabon' 'Gambia'
 'Ghana' 'Grenada' 'Guatemala' 'Guinea' 'Guinea-Bissau' 'Guyana' 'Haiti'
 'India' 'Indonesia' 'Iran' 'Iraq' 'Jamaica' 'Jordan' 'Kazakhstan' 'Kenya'
 'Kiribati' 'Kyrgyz Republic' 'Lao' 'Lesotho' 'Liberia' 'Madagascar'
 'Malawi' 'Mali' 'Mauritania' 'Micronesia, Fed. Sts.' 'Mongolia' 'Morocco'
 'Mozambique' 'Myanmar' 'Namibia' 'Nepal' 'Niger' 'Nigeria' 'Pakistan'
 'Paraguay' 'Peru' 'Philippines' 'Russia' 'Rwanda' 'Samoa' 'Senegal'
 'Sierra Leone' 'Solomon Islands' 'South Africa' 'Sri Lanka'
 'St. Vincent and the Grenadines' 'Sudan' 'Suriname' 'Tajikistan'
 'Tanzania' 'Timor-Leste' 'Togo' 'Tonga

In [112]:
print(segmen1["country"].unique())

['Albania' 'Antigua and Barbuda' 'Australia' 'Austria' 'Bahamas' 'Bahrain'
 'Barbados' 'Belarus' 'Belgium' 'Bosnia and Herzegovina' 'Brazil' 'Brunei'
 'Bulgaria' 'Canada' 'Chile' 'Colombia' 'Costa Rica' 'Croatia' 'Cyprus'
 'Czech Republic' 'Denmark' 'Estonia' 'Finland' 'France' 'Georgia'
 'Germany' 'Greece' 'Hungary' 'Iceland' 'Ireland' 'Israel' 'Italy' 'Japan'
 'Kuwait' 'Latvia' 'Lebanon' 'Libya' 'Lithuania' 'Luxembourg'
 'Macedonia, FYR' 'Malaysia' 'Maldives' 'Malta' 'Mauritius' 'Moldova'
 'Montenegro' 'Netherlands' 'New Zealand' 'Norway' 'Oman' 'Panama'
 'Poland' 'Portugal' 'Qatar' 'Romania' 'Saudi Arabia' 'Serbia'
 'Seychelles' 'Singapore' 'Slovak Republic' 'Slovenia' 'South Korea'
 'Spain' 'Sweden' 'Switzerland' 'Thailand' 'Tunisia' 'Turkey' 'Ukraine'
 'United Arab Emirates' 'United Kingdom' 'United States' 'Uruguay'
 'Vietnam']


In [113]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

data = pd.read_csv("country_data.csv")

X = data.drop('country', axis=1) 
country_labels = data['country']

pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())
])

X_scaled = pipeline.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

best_k = 2
best_score = 0
rentang_k = range(2, 6)

print("\nEvaluasi K:")
for k in rentang_k:
    model = KMeans(n_clusters=k, random_state=42, n_init='auto')
    label = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, label)
    print(f"K={k}, Silhouette Score: {score:.4f}")
    
    if score > best_score:
        best_score = score
        best_k = k

print(f"\nJumlah Cluster Terbaik (K): {best_k} dengan Silhouette Score: {best_score:.4f}")

final_model = KMeans(n_clusters=best_k, random_state=42, n_init='auto')
data['Segmen'] = final_model.fit_predict(X_scaled)

print("\nHasil Clustering:")
print(data[['country', 'Segmen']].head(10))


Evaluasi K:
K=2, Silhouette Score: 0.2663
K=3, Silhouette Score: 0.2856
K=4, Silhouette Score: 0.2880
K=5, Silhouette Score: 0.3009

Jumlah Cluster Terbaik (K): 5 dengan Silhouette Score: 0.3009

Hasil Clustering:
               country  Segmen
0          Afghanistan       2
1              Albania       0
2              Algeria       0
3               Angola       2
4  Antigua and Barbuda       0
5            Argentina       0
6              Armenia       0
7            Australia       1
8              Austria       1
9           Azerbaijan       0


In [114]:
print(data["Segmen"].unique())

[2 0 1 4 3]


In [115]:
segmen0 = data.loc[data["Segmen"] == 0]
print(segmen0.describe())
print(segmen0["country"].unique())

       child_mort    exports     health     imports        income  inflation  \
count   87.000000  87.000000  87.000000   87.000000     87.000000  87.000000   
mean    22.219540  41.083782   6.246667   47.795010  12678.965517   7.727747   
std     14.140531  19.330113   2.168250   20.071805   8644.118451   7.762552   
min      4.500000   0.109000   1.970000    0.065900   1780.000000  -4.210000   
25%     11.350000  26.900000   4.885000   32.700000   6705.000000   2.485000   
50%     18.600000  38.400000   5.980000   49.200000  10500.000000   5.990000   
75%     29.150000  51.500000   7.525000   60.550000  16800.000000  10.050000   
max     64.400000  93.800000  14.200000  108.000000  45400.000000  45.900000   

       life_expec  total_fer          gdpp  Segmen  
count   87.000000  87.000000     87.000000    87.0  
mean    72.625287   2.333333   6494.390805     0.0  
std      4.304952   0.733359   4747.414933     0.0  
min     57.100000   1.250000    592.000000     0.0  
25%     69.900

In [116]:
segmen1 = data.loc[data["Segmen"] == 1]
print(segmen1.describe())
print(segmen1["country"].unique())

       child_mort     exports     health    imports         income  inflation  \
count    32.00000   32.000000  32.000000  32.000000      32.000000  32.000000   
mean      5.18125   46.118750   9.088438  40.584375   44021.875000   2.513844   
std       2.62868   21.419714   3.174913  17.380952   19749.115691   4.225203   
min       2.60000   12.400000   1.810000  13.600000   22900.000000  -3.220000   
25%       3.75000   28.875000   7.887500  28.075000   32450.000000   0.343000   
50%       4.20000   44.250000   9.510000  37.250000   40100.000000   1.105000   
75%       5.30000   64.075000  11.000000  49.175000   45550.000000   3.175000   
max      13.80000  103.000000  17.900000  86.500000  125000.000000  16.700000   

       life_expec  total_fer          gdpp  Segmen  
count   32.000000  32.000000     32.000000    32.0  
mean    80.081250   1.788437  42118.750000     1.0  
std      1.874177   0.360958  15347.237589     0.0  
min     73.800000   1.230000  19800.000000     1.0  
25%  

In [117]:
segmen2 = data.loc[data["Segmen"] == 2]
print(segmen2.describe())
print(segmen2["country"].unique())

       child_mort    exports     health     imports       income  inflation  \
count   44.000000  44.000000  44.000000   44.000000     44.00000  44.000000   
mean    94.313636  28.677500   6.329091   42.872727   3503.25000   9.895568   
std     32.352474  18.572688   2.712319   17.837801   5477.15483   7.433764   
min     53.700000   2.200000   2.200000   17.200000    609.00000   0.885000   
25%     66.075000  16.725000   4.502500   29.900000   1380.00000   3.792500   
50%     90.250000  23.300000   5.480000   41.500000   1835.00000   8.215000   
75%    111.000000  36.850000   7.715000   49.375000   3290.00000  16.225000   
max    208.000000  85.800000  13.100000  101.000000  33700.00000  26.500000   

       life_expec  total_fer          gdpp  Segmen  
count   44.000000  44.000000     44.000000    44.0  
mean    59.022727   5.047727   1753.909091     2.0  
std      6.538610   1.016093   2950.405544     0.0  
min     32.100000   2.590000    231.000000     2.0  
25%     56.450000   4.5

In [118]:
segmen3 = data.loc[data["Segmen"] == 3]
print(segmen3.describe())
print(segmen3["country"].unique())

       child_mort  exports  health  imports  income  inflation  life_expec  \
count         1.0      1.0    1.00      1.0     1.0        1.0         1.0   
mean        130.0     25.3    5.07     17.4  5150.0      104.0        60.5   
std           NaN      NaN     NaN      NaN     NaN        NaN         NaN   
min         130.0     25.3    5.07     17.4  5150.0      104.0        60.5   
25%         130.0     25.3    5.07     17.4  5150.0      104.0        60.5   
50%         130.0     25.3    5.07     17.4  5150.0      104.0        60.5   
75%         130.0     25.3    5.07     17.4  5150.0      104.0        60.5   
max         130.0     25.3    5.07     17.4  5150.0      104.0        60.5   

       total_fer    gdpp  Segmen  
count       1.00     1.0     1.0  
mean        5.84  2330.0     3.0  
std          NaN     NaN     NaN  
min         5.84  2330.0     3.0  
25%         5.84  2330.0     3.0  
50%         5.84  2330.0     3.0  
75%         5.84  2330.0     3.0  
max         5.84 

In [119]:
segmen4 = data.loc[data["Segmen"] == 4]
print(segmen4.describe())
print(segmen4["country"].unique())

       child_mort     exports    health     imports        income  inflation  \
count    3.000000    3.000000  3.000000    3.000000      3.000000   3.000000   
mean     4.133333  176.000000  6.793333  156.666667  64033.333333   2.468000   
std      2.309401   23.515952  2.492877   16.165808  32460.642836   2.179718   
min      2.800000  153.000000  3.960000  142.000000  28300.000000  -0.046000   
25%      2.800000  164.000000  5.865000  148.000000  50200.000000   1.787000   
50%      2.800000  175.000000  7.770000  154.000000  72100.000000   3.620000   
75%      4.800000  187.500000  8.210000  164.000000  81900.000000   3.725000   
max      6.800000  200.000000  8.650000  174.000000  91700.000000   3.830000   

       life_expec  total_fer           gdpp  Segmen  
count    3.000000   3.000000       3.000000     3.0  
mean    81.433333   1.380000   57566.666667     4.0  
std      1.205543   0.240624   43011.665084     0.0  
min     80.300000   1.150000   21100.000000     4.0  
25%     8

Time series

In [120]:
import pandas as pd
import numpy as np

df_time = pd.DataFrame({'Tanggal': pd.date_range('2026-01-01', periods=3),
                        'Target': [139, 128, 172]})
df_time = df_time.set_index('Tanggal')

df_time['Time_Dummy'] = np.arange(len(df_time.index))
print(df_time)

            Target  Time_Dummy
Tanggal                       
2026-01-01     139           0
2026-01-02     128           1
2026-01-03     172           2


In [121]:
df_time['Lag_1'] = df_time['Target'].shift(1)
print(df_time)

            Target  Time_Dummy  Lag_1
Tanggal                              
2026-01-01     139           0    NaN
2026-01-02     128           1  139.0
2026-01-03     172           2  128.0


In [122]:
from sklearn.model_selection import train_test_split

df_clean = df_time.dropna()

X = df_clean[['Time_Dummy', 'Lag_1']]
y = df_clean['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print("X_train: \n", X_train, "\nY_train : \n", y_train)
print("X_test: \n", X_test, "\nY_test : \n", y_test)

X_train: 
             Time_Dummy  Lag_1
Tanggal                      
2026-01-02           1  139.0 
Y_train : 
 Tanggal
2026-01-02    128
Name: Target, dtype: int64
X_test: 
             Time_Dummy  Lag_1
Tanggal                      
2026-01-03           2  128.0 
Y_test : 
 Tanggal
2026-01-03    172
Name: Target, dtype: int64
